# Notebook 02 — NetFlow Anomaly Investigator with RAG

> **Use case:** A flow collector raises an alert — *traffic on `core-1:eth2` jumped 12× in 5 minutes*. Instead of paging a human, an agent retrieves the most relevant **runbooks** from a corpus, summarises the likely cause, and cites its sources.

**What you will learn**
- How to build a **small RAG pipeline** end-to-end with just `scikit-learn` + `numpy`.
- The difference between **lexical (BM25)** and **semantic (TF-IDF/cosine)** retrieval.
- How to combine them with **Reciprocal Rank Fusion (RRF)**.
- How to evaluate retrieval with **Recall@k**.

The notebook is fully offline — no external API needed.

> *Why TF-IDF and not real embeddings?* Real embeddings (e.g. `sentence-transformers`) would download ~100 MB and need a GPU for speed. TF-IDF is a *valid* sparse embedding that illustrates the same RAG mechanics. Section 7 explains how to swap it.

## 1. Synthetic runbook corpus

We pretend the NOC has 10 short runbooks. Each one has an `id`, a `title`, a `body`, and an authoritative `cause_tag` we will use later for evaluation.

In [ ]:
CORPUS = [
    {"id": "RB001", "cause_tag": "ddos",
     "title": "Volumetric DDoS playbook",
     "body": ("Symptoms: sudden traffic spike on edge interfaces, many small flows from "
              "diverse source IPs, unusually high packets-per-second to a single victim. "
              "Action: enable upstream BGP flowspec or RTBH for the victim prefix.")},
    {"id": "RB002", "cause_tag": "backup_window",
     "title": "Scheduled backup window traffic",
     "body": ("Every weekday between 02:00 and 04:00 UTC the storage cluster pushes incremental "
              "backups to the offsite datacentre. Expect 10x-20x throughput increase on the "
              "core uplinks during this window. Not an anomaly.")},
    {"id": "RB003", "cause_tag": "elephant_flow",
     "title": "Elephant flows from research cluster",
     "body": ("The HPC research cluster occasionally transfers multi-TB datasets to external "
              "partners. A single flow can saturate a 10G link for tens of minutes. Identify "
              "with NetFlow top-talkers and notify the science DMZ team.")},
    {"id": "RB004", "cause_tag": "misconfig",
     "title": "Routing loop after recent change",
     "body": ("Traffic spikes following a configuration change often indicate a routing loop. "
              "Check TTL distribution: many low-TTL packets between the same two IPs is a strong "
              "indicator. Compare against the change log of the last 60 minutes.")},
    {"id": "RB005", "cause_tag": "malware_c2",
     "title": "Possible malware command-and-control",
     "body": ("Persistent low-rate beaconing from a single internal host to a known-bad external "
              "IP is suspicious. Correlate with threat intelligence feed; if confirmed, isolate "
              "the host via NAC quarantine VLAN.")},
    {"id": "RB006", "cause_tag": "link_failure",
     "title": "Link failure causing rerouting spike",
     "body": ("When a primary link fails, traffic reroutes to the backup path which then sees a "
              "sudden, sustained increase. Verify with interface counters and IGP convergence "
              "events. Not malicious unless capacity is now exceeded.")},
    {"id": "RB007", "cause_tag": "backup_window",
     "title": "Storage replication windows",
     "body": ("Block storage replication runs on weekends. Plan windows: Saturday 22:00 to Sunday "
              "06:00 UTC. Saturation of core links during this period is expected.")},
    {"id": "RB008", "cause_tag": "flash_crowd",
     "title": "Flash crowd from marketing campaign",
     "body": ("Marketing announces new product launches via push notification. Expect a flash crowd "
              "of legitimate users hitting the public website within minutes. Pre-warm the CDN; "
              "do NOT trigger DDoS mitigation.")},
    {"id": "RB009", "cause_tag": "ddos",
     "title": "Reflection / amplification attacks",
     "body": ("NTP, DNS and Memcached can be abused for reflection attacks. Signature: very large "
              "UDP responses to many random source ports of a single victim IP. Mitigate with "
              "upstream filtering on source port.")},
    {"id": "RB010", "cause_tag": "misconfig",
     "title": "BGP route leak from peer",
     "body": ("A peer accidentally advertised the full internet table to us, causing transit "
              "traffic to flow through our backbone. Detect via sudden AS-PATH length anomalies "
              "and unexpected traffic from external prefixes.")},
]

print(f"Loaded {len(CORPUS)} runbooks.")

## 2. Lexical retriever — BM25 from scratch

BM25 is the workhorse of search engines. It scores each document by how *unusually often* the query terms appear in it. We implement it in ~25 lines so there is no magic.

In [ ]:
import math, re
from collections import Counter

TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())

class BM25:
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75) -> None:
        self.k1, self.b = k1, b
        self.tokens  = [tokenize(d) for d in docs]
        self.lengths = [len(t) for t in self.tokens]
        self.avgdl   = sum(self.lengths) / len(self.tokens)
        self.tf      = [Counter(t) for t in self.tokens]
        df: Counter = Counter()
        for t in self.tokens:
            df.update(set(t))
        N = len(self.tokens)
        self.idf = {term: math.log(1 + (N - n + 0.5) / (n + 0.5))
                    for term, n in df.items()}

    def score(self, query: str) -> list[float]:
        q = tokenize(query)
        scores = [0.0] * len(self.tf)
        for term in q:
            idf = self.idf.get(term)
            if idf is None:
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(term, 0)
                if f == 0:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * self.lengths[i] / self.avgdl)
                scores[i] += idf * (f * (self.k1 + 1)) / denom
        return scores

bm25 = BM25([d["title"] + " " + d["body"] for d in CORPUS])
print("BM25 indexed.")

## 3. Semantic retriever — TF-IDF + cosine

A TF-IDF vector is a (very) simple **sparse embedding**. With cosine similarity it captures documents that *share rare terms* with the query. We use `scikit-learn` because it is universally available.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

VECTORIZER = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
DOC_VECTORS = VECTORIZER.fit_transform([d["title"] + " " + d["body"] for d in CORPUS])

def tfidf_scores(query: str) -> np.ndarray:
    qv = VECTORIZER.transform([query])
    return cosine_similarity(qv, DOC_VECTORS)[0]

print("TF-IDF index shape:", DOC_VECTORS.shape)

## 4. Hybrid retrieval with Reciprocal Rank Fusion

Neither retriever wins all the time:
- **BM25** is unbeatable on **exact terms** (`RTBH`, `flowspec`).
- **TF-IDF / embeddings** catch **paraphrases** (`rerouting spike` ≈ `traffic surge`).

RRF combines the two **ranked lists** (not raw scores) which is robust and parameter-light:

$$\text{RRF}(d) = \sum_{r} \frac{1}{k + \text{rank}_r(d)}$$

In [ ]:
def ranks_from_scores(scores) -> dict:
    """Return {doc_index: rank (1-based)} ordered by descending score."""
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return {idx: rank + 1 for rank, idx in enumerate(order)}

def retrieve(query: str, k: int = 3, mode: str = "hybrid") -> list:
    bm = list(bm25.score(query))
    tf = list(tfidf_scores(query))

    if mode == "bm25":
        ranking = ranks_from_scores(bm)
        ordered = sorted(ranking, key=ranking.get)
    elif mode == "tfidf":
        ranking = ranks_from_scores(tf)
        ordered = sorted(ranking, key=ranking.get)
    else:  # hybrid via RRF
        rb, rt = ranks_from_scores(bm), ranks_from_scores(tf)
        K = 60
        fused = {i: 1 / (K + rb[i]) + 1 / (K + rt[i]) for i in range(len(CORPUS))}
        ordered = sorted(fused, key=fused.get, reverse=True)

    return [CORPUS[i] for i in ordered[:k]]

for hit in retrieve("sudden traffic spike right after a config change", k=3):
    print(f"  {hit['id']}  {hit['title']}")

## 5. Investigating an anomaly

Given a NetFlow alert in plain English, we:
1. retrieve the top-3 runbooks,
2. format an **answer with citations** (no real LLM needed — a template suffices to illustrate),
3. flag the **most likely cause tag**.

In a production agent this template step would be done by an LLM with the retrieved chunks in its context (RAG).

In [ ]:
from collections import Counter

def investigate(alert: str, k: int = 3) -> dict:
    hits = retrieve(alert, k=k, mode="hybrid")
    tag_counts = Counter(h["cause_tag"] for h in hits)
    likely_cause, _ = tag_counts.most_common(1)[0]
    citations = ", ".join(h["id"] for h in hits)
    report = (
        f"ALERT: {alert}\n"
        f"LIKELY CAUSE: {likely_cause}\n"
        f"EVIDENCE (top {k} matches):\n" +
        "\n".join(f"  - [{h['id']}] {h['title']}" for h in hits) +
        f"\nCITATIONS: {citations}"
    )
    return {"alert": alert, "likely_cause": likely_cause, "hits": hits, "report": report}

for alert in [
    "Traffic on core-1 eth2 jumped 12x at 02:30 UTC, mostly to the offsite DC prefix.",
    "Many small UDP responses to random source ports targeting 198.51.100.7.",
    "Sustained low-rate beaconing from one internal workstation to an external IP.",
    "Web frontend traffic surged within 2 minutes of the product launch push.",
]:
    print(investigate(alert)["report"])
    print("-" * 70)

## 6. Evaluating retrieval — Recall@3

A RAG pipeline is only as good as its retriever. We label a small eval set of `(query, correct cause tag)` pairs and compute **Recall@k**: did at least one of the top-k retrieved runbooks have the correct tag?

In [ ]:
EVAL_SET = [
    ("Edge interface flooded with tiny packets from thousands of source IPs", "ddos"),
    ("Spike between 02 and 04 UTC every weekday on the core uplinks",         "backup_window"),
    ("Single 5 TB transfer saturates a 10G link for 20 minutes",              "elephant_flow"),
    ("Loop suspected after a config push 30 minutes ago, low TTL packets",    "misconfig"),
    ("Workstation talks to known-bad IP every 90 seconds, low rate",          "malware_c2"),
    ("Primary uplink dropped, backup now saturated",                          "link_failure"),
    ("Replication between datacentres maxes out the WAN every Saturday night","backup_window"),
    ("Marketing launched and the site is hammered by real users",             "flash_crowd"),
    ("Memcached reflection flood, huge UDP responses",                        "ddos"),
    ("AS path lengths look weird and we suddenly carry foreign prefixes",     "misconfig"),
]

def recall_at_k(mode: str, k: int = 3) -> float:
    hits = 0
    for query, gold in EVAL_SET:
        retrieved = retrieve(query, k=k, mode=mode)
        if any(r["cause_tag"] == gold for r in retrieved):
            hits += 1
    return hits / len(EVAL_SET)

print(f"Recall@3  BM25-only : {recall_at_k('bm25'):.0%}")
print(f"Recall@3  TFIDF-only: {recall_at_k('tfidf'):.0%}")
print(f"Recall@3  Hybrid RRF: {recall_at_k('hybrid'):.0%}")

## 7. Going further — real embeddings

To swap TF-IDF for **real semantic embeddings** with almost no code change:

```python
from sentence_transformers import SentenceTransformer
MODEL = SentenceTransformer("BAAI/bge-small-en-v1.5")
DOC_VECTORS = MODEL.encode([d["title"] + " " + d["body"] for d in CORPUS],
                           normalize_embeddings=True)

def tfidf_scores(query):
    qv = MODEL.encode([query], normalize_embeddings=True)
    return (DOC_VECTORS @ qv.T).flatten()
```

For production:
- Use a vector DB (Chroma / Qdrant / pgvector) instead of in-memory arrays.
- Add a **cross-encoder re-ranker** on the top-20 hits.
- Wrap the answer-formatting step with a real LLM, and require it to cite runbook IDs verbatim (faithfulness).
- Log every query, retrieval, and answer to your tracing tool (Phoenix / LangSmith) — see Chapter 11.

## Exercises

1. **Adversarial query.** Find an alert that fools the hybrid retriever (wrong top-1). Add a runbook that fixes it.
2. **Chunking.** Split each runbook body into sentences and index those instead. Does Recall@3 change?
3. **Faithfulness.** Modify `investigate` so that the report only mentions a cause if **at least 2** of the top-3 retrieved runbooks share the same `cause_tag` — otherwise it says `unknown`.
4. **Cross-encoder.** Pip-install `sentence-transformers`, plug a re-ranker (`ms-marco-MiniLM-L-6-v2`) on the top-5 of the hybrid result. Re-measure Recall@3 *and* report Recall@1.
5. **LLM answer.** Replace the template in §5 by an Ollama call (`llama3.1:8b`) that takes the retrieved chunks and produces a 3-line summary with citations.